# Chapter 5 — The Honor That Flips
### Orthogonal Procrustes Space Shifts

This notebook aligns the semantic space of the first half (chapters 1–5) and second half (chapters 6–10) of Esther using Orthogonal Procrustes rotation, mapping which words changed meaning most.

In [ ]:
from esther_nlp.data import load_chapters_and_verses
from esther_nlp.tokenizer import EstherBpeTokenizer, compute_vocabulary_counts
from esther_nlp.embeddings import train_word2vec, align_spaces_procrustes
from esther_nlp.plots import fig_05a_top_movers, fig_05b_neighbor_clouds
from pathlib import Path
from IPython.display import Image, display

chapters, all_verses = load_chapters_and_verses()
tokenizer = EstherBpeTokenizer(vocab_size=1000)
tokenizer.train(all_verses)
tokenized_verses = [tokenizer.encode_to_tokens(v) for v in all_verses]
counts_total = compute_vocabulary_counts(tokenized_verses)

first_half_v  = [v for ch in chapters[:5] for v in ch]
second_half_v = [v for ch in chapters[5:] for v in ch]

def get_tokens(texts):
    tk = EstherBpeTokenizer(vocab_size=1000)
    tk.train(texts)
    return [tk.encode_to_tokens(v) for v in texts]

# Train embedding spaces on each half separately
m1 = train_word2vec(get_tokens(first_half_v),  sg=1, epochs=100, seed=42, vector_size=50)
m2 = train_word2vec(get_tokens(second_half_v), sg=1, epochs=100, seed=42, vector_size=50)

# Align spaces using Orthogonal Procrustes rotation
shifts, _, _, _ = align_spaces_procrustes(m1, m2, counts_total, min_corpus_freq=3)

# Generate top 10 movers bar chart
top_10 = shifts[:10]
movers_tokens = [w for w, _ in top_10]
movers_values = [v for _, v in top_10]

fig_movers_path = Path("fig_05a_top_movers.png")
fig_05a_top_movers(movers_tokens, movers_values, fig_movers_path)
display(Image(filename=fig_movers_path))

# Generate neighbor tag clouds
n1 = m1.wv.most_similar("יקר", topn=10) if "יקר" in m1.wv else []
n2 = m2.wv.most_similar("יקר", topn=10) if "יקר" in m2.wv else []
fig_clouds_path = Path("fig_05b_neighbor_clouds.png")
fig_05b_neighbor_clouds(n1, n2, fig_clouds_path)
display(Image(filename=fig_clouds_path))

## Semantic Shifts Analysis & Conclusion

### Core Observations
1. **`יקר` is the Top Mover**: The token `יקר` (yakar / honor / glory) is the single most mobile word in the entire vocabulary between halves, undergoing a drastic spatial rotation.
2. **Neighborhood Shift**: 
   - **First Half**: `יקר` sits surrounded by standard court-pomp and royal decree terms: `כבוד` (glory), `קטן` (small), `למגדול` (to great).
   - **Second Half**: `יקר` migrates to the center of Chapter 6's ironical honors parade: `למרדכי` (to Mordecai), `נעשה` (was done), `עמו` (with him).

### Narrative and Literary Conclusion
`יקר` acts as the lexical atom of the book's turning point (*ונהפוך הוא*). In the first half, it is a tool of institutional royal decrees (decorating the Persian court). In the center of the book (Chapter 6), it becomes a highly ironical weapon, designed by Haman for himself, but forced to be delivered to Mordecai instead. In the end (Chapter 8), it is transferred to the Jewish collective (*ליהודים היתה אורה ושמחה וששן ויקר*). The word literally travels across the book's narrative vector, transforming from royal pomp, to biting irony, to national salvation. The Procrustes alignment tracks this semantic voyage cleanly.